In [1]:
import cv2
import numpy as np
import time
import chess
import chess.engine
import requests
import json

# Computer Vision Code

In [2]:
def crop_by_percentage(image, percent):
        h, w = image.shape[:2]
        dx = int(w * percent)
        dy = int(h * percent)
        cropped = image[dy:h - dy, dx:w - dx]
        return cropped

def get_dots(image, DEBUG=False):
    # Define two HLS color ranges
    lower = np.array([115, 50, 70])
    upper = np.array([140, 230, 255])
    
    # Create masks
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HLS)
    colorMask = cv2.inRange(hsv, lower, upper)

    if DEBUG:
        cv2.imshow('Color Mask', colorMask)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()
    
    # Find contours in binary image
    contours, _ = cv2.findContours(colorMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Get center points of valid contours
    centers = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 10:
            M = cv2.moments(cnt)
            if M['m00'] != 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                centers.append((cx, cy))

    return centers
    # Proceed if exactly 4 red dots found
    #if len(centers) != 4:
    #    raise Exception("not 4")

def divide_into_grid(image2, shave=0.1):
    h, w = image2.shape[:2]
    
    grid = []
    for y in range(8):
        for x in range(8):
            x1 = (x * w) // 8
            y1 = (y * h) // 8
            x2 = x1 + (w//8)
            y2 = y1 + (h//8)
            cell = image2[y1:y2, x1:x2]
            cell = crop_by_percentage(cell, shave)
            grid.append(cell)
    return grid

In [3]:
def get_occupancy(board):
    occupancy = np.zeros(64, dtype=np.bool)
    for i in range(64):
        occupancy[i] = (board.piece_at(i) != None)
    
    occupancy = occupancy.reshape((8,8))
    occupancy = np.flip(occupancy, axis=0)
    return occupancy

def get_maskmap(board):
    maskmap = {}
    maskmap[""] = get_occupancy(board)
    for move in board.legal_moves:
        board.push(move)
        maskmap[move.uci()] = get_occupancy(board)
        board.pop()
    return maskmap

In [5]:
prevCrop = None

DEBUG = True

camera = cv2.VideoCapture(0)

def try_vision(playboard, aiwhite):
    time.sleep(1)
    # Load image and apply median blur
    #image = cv2.imread('image.jpg')
    #image = cv2.medianBlur(image, 5)
    _, image = camera.read()
    
    if DEBUG:
        cv2.imshow('Raw', image)
        cv2.waitKey(15000)
        cv2.destroyAllWindows()
    
    centers = get_dots(image, DEBUG)

    if len(centers) != 4:
        if prevCrop != None:
            centers = prevCrop
        else:
            print("Crop error", len(centers))
            return ""

    prevCrop = centers
    
    pts = np.array(centers, dtype='float32')
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    ordered = np.array([
        pts[np.argmin(s)],       # top-left
        pts[np.argmin(diff)],    # top-right
        pts[np.argmax(s)],       # bottom-right
        pts[np.argmax(diff)]     # bottom-left
    ])
    
    width = int(max(np.linalg.norm(ordered[0] - ordered[1]), np.linalg.norm(ordered[2] - ordered[3])))
    height = int(max(np.linalg.norm(ordered[0] - ordered[3]), np.linalg.norm(ordered[1] - ordered[2])))
    dst_pts = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype='float32')
    
    matrix = cv2.getPerspectiveTransform(ordered, dst_pts)
    cropped = cv2.warpPerspective(image, matrix, (width, height))
    
    cropped = crop_by_percentage(cropped, 0.02)

    
    # Display the result
    if DEBUG:
        cv2.imshow('Cropped Area', cropped)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()

    canny = cv2.Canny(cropped, 120, 240)
    
    if DEBUG:
        cv2.imshow('Cropped Area', canny)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()

    # Example usage
    grid_images = divide_into_grid(canny)
    
    arr = np.array(grid_images)
    arr = arr.reshape(8, 8, arr.shape[1] * arr.shape[2])
    
    arr = np.sum(arr, axis=2)

    if not aiwhite():
        arr = np.flip(arr, axis=0)
    #print(arr)
    
    #for i in range(64):
        # Show first cell as example
        #print(np.sum(grid_images[i]))
        #cv2.imshow('First Cell', grid_images[i])
        #cv2.waitKey(5000)
        #cv2.destroyAllWindows()

    playmasks = get_maskmap(playboard)

    bestscore = -99999999999
    bestmove = ""
    
    for key, value in playmasks.items():
        on_right = np.sum(arr[value]).astype(np.int64)
        off_wrong = np.sum(arr[~value]).astype(np.int64)
        score = on_right - off_wrong
        if score > bestscore:
            bestscore = score
            bestmove = key

    print("vision", bestscore, bestmove)
    return bestmove
        

[ WARN:0@42.008] global cap_gstreamer.cpp:1173 isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
[ WARN:0@42.008] global cap_v4l.cpp:913 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@42.113] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range


# Robot Arm Code 

In [24]:
def movearm(x,y,z=10,spd=0.25,t=3.14):
	requests.get("http://192.168.4.1/js?json=" + json.dumps({"T":104,"x":x,"y":y,"z":z,"t":t,"spd":spd}))

#      X+
#      
#  B-------D
#  |       |
#  |       |  Y-
#  |       |
#  A-------C
#     ARM
#      |
#     Y=0
A = (80,  160, -35)
B = (430, 155, -10)
C = (70, -165, -35)
D = (425,-170, -10)

# Each chess square roughly 45 wide

# air, pawn, knight, bishop, rook, queen, king
HEIGHTS = [80, -30, -20, -10, -20, 5, 5, 100]
CLOSED = 2.9
OPEN = 2.3

hdrop = 2.5


def movesq(sq, piece=0, heightOffset=0, spd=0.25,t=CLOSED, shift=0, shiftForward=0):
	offset = HEIGHTS[piece] + heightOffset


	dy = (ord(sq[0]) - ord('a'))/7
	dx = (ord(sq[1]) - ord('1'))/7
	dy = 1-(dy*2)

	if dy >= 0:
		x = (B[0]*dx) + (A[0]*(1-dx))
		y = (B[1]*dx) + (A[1]*(1-dx))
		z = (B[2]*dx) + (A[2]*(1-dx))
		y *= abs(dy)
	else:
		x = (D[0]*dx) + (C[0]*(1-dx))
		y = (D[1]*dx) + (C[1]*(1-dx))
		z = (D[2]*dx) + (C[2]*(1-dx))
		y *= abs(dy)

	if dx == 0:
		theta = np.arctan((0.5*dy)/0.00001) + (np.pi / 2)
	else:
		theta = np.arctan((0.5*dy)/dx) + (np.pi / 2)

	x += shift*np.cos(theta)
	y += shift*np.sin(theta)

	shiftForward *= np.sqrt(dx**2 + (dy/2)**2)/1.5

	x += shiftForward * np.sin(theta)
	y += shiftForward * -np.cos(theta)

	movearm(x, y, z=z+offset, spd=spd, t=t)


def calibrate():
	movearm(A[0],A[1],A[2])
	input()
	movearm(B[0],B[1], B[2])
	input()
	movearm(C[0],C[1],C[2])
	input()
	movearm(D[0],D[1],D[2])
	input()




def movemove(movestr, piecetype=1):
	start = movestr[0:2]  
	end = movestr[2:4]
	movesq(start,0,t=OPEN,spd=0.25,shift=25,shiftForward=0)
	time.sleep(1)
	movesq(start,piecetype,t=OPEN,spd=0.1,shift=25,shiftForward=0)
	time.sleep(1)
	movesq(start,piecetype,t=CLOSED,spd=0.1,shift=25,shiftForward=0)
	time.sleep(1)
	movesq(start,0,t=CLOSED,shift=25,shiftForward=0)
	time.sleep(1)
	movesq(end,0,t=CLOSED,shift=10, shiftForward=0)
	time.sleep(1)
	movesq(end,piecetype,heightOffset=hdrop,t=CLOSED,spd=0.1,shift=10, shiftForward=0)
	time.sleep(1)
	movesq(end,piecetype,heightOffset=hdrop,t=OPEN,spd=0.1,shift=10, shiftForward=0)
	time.sleep(1)
	movesq(end,0,t=CLOSED,shift=10, shiftForward=0)

def calibrate2():
	movesq("h1", 0)
	input()

	movemove("a2a4",1)
	movemove("e2e4",1)
	movemove("h2h4",1)
	movemove("a7a5",1)
	movemove("e7e5",1)
	movemove("h7h5",1)

	movemove("g1f3",2)
	movemove("b8c6",3)

	movemove("f1b5",3)
	movemove("g8f6",2)

	movemove("e1g1",6)
	movemove("h1f1",4)

	movemove("d8d4",5)
	movemove("d1d5",5)

In [25]:
#calibrate()

In [33]:
#calibrate2()
movesq("j1",0)

# Manager Code

In [35]:
engine = chess.engine.SimpleEngine.popen_uci("./main")

STARTPOS = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"

manual = True
startfen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
moves = ['g1f3', 'g8f6', 'g2g3', 'd7d5', 'f1g2', 'c7c5', 'c2c4', 'e7e6', 'e1g1', 'd5d4', 'e2e3', 'f8e7', 'a2a3', 'a7a5', 'e3d4', 'c5d4', 'b2b3', 'e8g8', 'f1e1', 'b8c6', 'd2d3', 'e7d6', 'f3g5', 'h7h6', 'g5e4', 'f6e4', 'g2e4', 'f7f5', 'e4g2', 'd8b6', 'f2f4', 'c8d7', 'b1d2', 'f8f6', 'a1b1', 'g7g5', 'd2f3', 'g5f4', 'c1f4', 'd6f4', 'g3f4', 'g8h8', 'd1d2', 'a8g8', 'g1h1', 'f6g6', 'e1e2', 'c6d8', 'f3e5', 'g6g7', 'b3b4', 'b6d6', 'c4c5', 'd6c7', 'd2b2', 'd8c6', 'e5d7', 'c7d7', 'b4a5', 'c6a5', 'g2f3', 'a5c6', 'e2g2', 'd7c7', 'b2d2', 'c6d8', 'g2g7', 'g8g7', 'b1c1', 'h6h5', 'c1c4', 'd8c6', 'c4a4', 'h5h4', 'a4a8', 'h8h7', 'd2d1', 'c6d8', 'd1c1', 'g7g6', 'c1f1', 'g6g8', 'f1c1', 'g8g7', 'c1f1', 'd8f7', 'a8a7', 'c7f4', 'a7b7', 'f7e5', 'b7g7', 'h7g7', 'f3e2', 'f4d2', 'a3a4', 'h4h3', 'a4a5', 'd2a5', 'f1h3']

aiwhite = False

def flip(s):
    print(s)
    
    if aiwhite:
        return s
    
    o = ""
    for i in range(len(s) // 2):
        sqstr = s[i*2:(i+1)*2]
        o += "hgfedcba"[(ord(sqstr[0]) - ord('a'))]
        o += "87654321"[(ord(sqstr[1]) - ord('1'))]
    return o

while True:
    print(startfen, moves)
    board = chess.Board(startfen)
    for ucimove in moves:
        move = chess.Move.from_uci(ucimove)
        board.push(move)
    print(board)

    nextmovestr = ""

    if (board.turn == chess.WHITE) == aiwhite:
        print("engine")
        nextmove = engine.play(board, chess.engine.Limit(time=2.5)).move
        nextmovestr = str(nextmove)
        
        capture = board.piece_at(nextmove.to_square) != None
        enpassant = board.is_en_passant(nextmove)
        castlek = board.is_kingside_castling(nextmove)
        castleq = board.is_queenside_castling(nextmove)
        promote = (nextmove.promotion != None)

        armmoves = []

        mover = int(board.piece_at(nextmove.from_square).piece_type)
    
        if capture:
            victim = int(board.piece_at(nextmove.to_square).piece_type)
            armmoves.append((victim,flip(chess.SQUARE_NAMES[nextmove.to_square])+"j1"))
        if enpassant:
            epoffset = 8
            if aiwhite:
                epoffset = -8
            
            armmoves.append((1,flip(chess.SQUARE_NAMES[board.ep_square + epoffset])+"j1"))

        if not promote:
            armmoves.append((mover,flip(nextmovestr)))
        else:
            armmoves.append((1,flip(nextmovestr[0:2]) + "j1"))
            print("PROMOTION", nextmovestr[2:4])

        if castlek:
            if aiwhite:
                armmoves.append((4,"h1f1"))
            else:
                armmoves.append((4,"a1c1"))
        if castleq:
            if aiwhite:
                armmoves.append((4,"a1d1"))
            else:
                armmoves.append((4,"h1e1"))
        
        for am in armmoves:
            print(am)
            piecet, moveu = am
            movemove(moveu, piecet)
        movesq("j1", 0)

    else:
        if manual:
            nextmovestr = input("ucimove:")
            if nextmovestr.startswith("undo"):
                moves.pop()
                moves.pop()
                continue
            if nextmovestr == "":
                break
        else:
            nextmovestr = try_vision(board, aiwhite)
            if nextmovestr == "":
                time.sleep(1)
                continue

    moves.append(nextmovestr)

rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 ['g1f3', 'g8f6', 'g2g3', 'd7d5', 'f1g2', 'c7c5', 'c2c4', 'e7e6', 'e1g1', 'd5d4', 'e2e3', 'f8e7', 'a2a3', 'a7a5', 'e3d4', 'c5d4', 'b2b3', 'e8g8', 'f1e1', 'b8c6', 'd2d3', 'e7d6', 'f3g5', 'h7h6', 'g5e4', 'f6e4', 'g2e4', 'f7f5', 'e4g2', 'd8b6', 'f2f4', 'c8d7', 'b1d2', 'f8f6', 'a1b1', 'g7g5', 'd2f3', 'g5f4', 'c1f4', 'd6f4', 'g3f4', 'g8h8', 'd1d2', 'a8g8', 'g1h1', 'f6g6', 'e1e2', 'c6d8', 'f3e5', 'g6g7', 'b3b4', 'b6d6', 'c4c5', 'd6c7', 'd2b2', 'd8c6', 'e5d7', 'c7d7', 'b4a5', 'c6a5', 'g2f3', 'a5c6', 'e2g2', 'd7c7', 'b2d2', 'c6d8', 'g2g7', 'g8g7', 'b1c1', 'h6h5', 'c1c4', 'd8c6', 'c4a4', 'h5h4', 'a4a8', 'h8h7', 'd2d1', 'c6d8', 'd1c1', 'g7g6', 'c1f1', 'g6g8', 'f1c1', 'g8g7', 'c1f1', 'd8f7', 'a8a7', 'c7f4', 'a7b7', 'f7e5', 'b7g7', 'h7g7', 'f3e2', 'f4d2', 'a3a4', 'h4h3', 'a4a5', 'd2a5', 'f1h3']
. . . . . . . .
. . . . . . k .
. . . . p . . .
q . P . n p . .
. . . p . . . .
. . . P . . . Q
. . . . B . . P
. . . . . . . K
engine
c5
a5c5
(1, 'f4

ucimove: h3g3


rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 ['g1f3', 'g8f6', 'g2g3', 'd7d5', 'f1g2', 'c7c5', 'c2c4', 'e7e6', 'e1g1', 'd5d4', 'e2e3', 'f8e7', 'a2a3', 'a7a5', 'e3d4', 'c5d4', 'b2b3', 'e8g8', 'f1e1', 'b8c6', 'd2d3', 'e7d6', 'f3g5', 'h7h6', 'g5e4', 'f6e4', 'g2e4', 'f7f5', 'e4g2', 'd8b6', 'f2f4', 'c8d7', 'b1d2', 'f8f6', 'a1b1', 'g7g5', 'd2f3', 'g5f4', 'c1f4', 'd6f4', 'g3f4', 'g8h8', 'd1d2', 'a8g8', 'g1h1', 'f6g6', 'e1e2', 'c6d8', 'f3e5', 'g6g7', 'b3b4', 'b6d6', 'c4c5', 'd6c7', 'd2b2', 'd8c6', 'e5d7', 'c7d7', 'b4a5', 'c6a5', 'g2f3', 'a5c6', 'e2g2', 'd7c7', 'b2d2', 'c6d8', 'g2g7', 'g8g7', 'b1c1', 'h6h5', 'c1c4', 'd8c6', 'c4a4', 'h5h4', 'a4a8', 'h8h7', 'd2d1', 'c6d8', 'd1c1', 'g7g6', 'c1f1', 'g6g8', 'f1c1', 'g8g7', 'c1f1', 'd8f7', 'a8a7', 'c7f4', 'a7b7', 'f7e5', 'b7g7', 'h7g7', 'f3e2', 'f4d2', 'a3a4', 'h4h3', 'a4a5', 'd2a5', 'f1h3', 'a5c5', 'h3g3']
. . . . . . . .
. . . . . . k .
. . . . p . . .
. . q . n p . .
. . . p . . . .
. . . P . . Q .
. . . . B . . P
. . . . . . . K
engine

ucimove: g3h4


rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 ['g1f3', 'g8f6', 'g2g3', 'd7d5', 'f1g2', 'c7c5', 'c2c4', 'e7e6', 'e1g1', 'd5d4', 'e2e3', 'f8e7', 'a2a3', 'a7a5', 'e3d4', 'c5d4', 'b2b3', 'e8g8', 'f1e1', 'b8c6', 'd2d3', 'e7d6', 'f3g5', 'h7h6', 'g5e4', 'f6e4', 'g2e4', 'f7f5', 'e4g2', 'd8b6', 'f2f4', 'c8d7', 'b1d2', 'f8f6', 'a1b1', 'g7g5', 'd2f3', 'g5f4', 'c1f4', 'd6f4', 'g3f4', 'g8h8', 'd1d2', 'a8g8', 'g1h1', 'f6g6', 'e1e2', 'c6d8', 'f3e5', 'g6g7', 'b3b4', 'b6d6', 'c4c5', 'd6c7', 'd2b2', 'd8c6', 'e5d7', 'c7d7', 'b4a5', 'c6a5', 'g2f3', 'a5c6', 'e2g2', 'd7c7', 'b2d2', 'c6d8', 'g2g7', 'g8g7', 'b1c1', 'h6h5', 'c1c4', 'd8c6', 'c4a4', 'h5h4', 'a4a8', 'h8h7', 'd2d1', 'c6d8', 'd1c1', 'g7g6', 'c1f1', 'g6g8', 'f1c1', 'g8g7', 'c1f1', 'd8f7', 'a8a7', 'c7f4', 'a7b7', 'f7e5', 'b7g7', 'h7g7', 'f3e2', 'f4d2', 'a3a4', 'h4h3', 'a4a5', 'd2a5', 'f1h3', 'a5c5', 'h3g3', 'g7h7', 'g3h4']
. . . . . . . .
. . . . . . . k
. . . . p . . .
. . q . n p . .
. . . p . . . Q
. . . P . . . .
. . . . B . . P
. . . 

ucimove: h4g3


rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 ['g1f3', 'g8f6', 'g2g3', 'd7d5', 'f1g2', 'c7c5', 'c2c4', 'e7e6', 'e1g1', 'd5d4', 'e2e3', 'f8e7', 'a2a3', 'a7a5', 'e3d4', 'c5d4', 'b2b3', 'e8g8', 'f1e1', 'b8c6', 'd2d3', 'e7d6', 'f3g5', 'h7h6', 'g5e4', 'f6e4', 'g2e4', 'f7f5', 'e4g2', 'd8b6', 'f2f4', 'c8d7', 'b1d2', 'f8f6', 'a1b1', 'g7g5', 'd2f3', 'g5f4', 'c1f4', 'd6f4', 'g3f4', 'g8h8', 'd1d2', 'a8g8', 'g1h1', 'f6g6', 'e1e2', 'c6d8', 'f3e5', 'g6g7', 'b3b4', 'b6d6', 'c4c5', 'd6c7', 'd2b2', 'd8c6', 'e5d7', 'c7d7', 'b4a5', 'c6a5', 'g2f3', 'a5c6', 'e2g2', 'd7c7', 'b2d2', 'c6d8', 'g2g7', 'g8g7', 'b1c1', 'h6h5', 'c1c4', 'd8c6', 'c4a4', 'h5h4', 'a4a8', 'h8h7', 'd2d1', 'c6d8', 'd1c1', 'g7g6', 'c1f1', 'g6g8', 'f1c1', 'g8g7', 'c1f1', 'd8f7', 'a8a7', 'c7f4', 'a7b7', 'f7e5', 'b7g7', 'h7g7', 'f3e2', 'f4d2', 'a3a4', 'h4h3', 'a4a5', 'd2a5', 'f1h3', 'a5c5', 'h3g3', 'g7h7', 'g3h4', 'h7g6', 'h4g3']
. . . . . . . .
. . . . . . . .
. . . . p . k .
. . q . n p . .
. . . p . . . .
. . . P . . Q .
. . . 

ucimove:  


rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 ['g1f3', 'g8f6', 'g2g3', 'd7d5', 'f1g2', 'c7c5', 'c2c4', 'e7e6', 'e1g1', 'd5d4', 'e2e3', 'f8e7', 'a2a3', 'a7a5', 'e3d4', 'c5d4', 'b2b3', 'e8g8', 'f1e1', 'b8c6', 'd2d3', 'e7d6', 'f3g5', 'h7h6', 'g5e4', 'f6e4', 'g2e4', 'f7f5', 'e4g2', 'd8b6', 'f2f4', 'c8d7', 'b1d2', 'f8f6', 'a1b1', 'g7g5', 'd2f3', 'g5f4', 'c1f4', 'd6f4', 'g3f4', 'g8h8', 'd1d2', 'a8g8', 'g1h1', 'f6g6', 'e1e2', 'c6d8', 'f3e5', 'g6g7', 'b3b4', 'b6d6', 'c4c5', 'd6c7', 'd2b2', 'd8c6', 'e5d7', 'c7d7', 'b4a5', 'c6a5', 'g2f3', 'a5c6', 'e2g2', 'd7c7', 'b2d2', 'c6d8', 'g2g7', 'g8g7', 'b1c1', 'h6h5', 'c1c4', 'd8c6', 'c4a4', 'h5h4', 'a4a8', 'h8h7', 'd2d1', 'c6d8', 'd1c1', 'g7g6', 'c1f1', 'g6g8', 'f1c1', 'g8g7', 'c1f1', 'd8f7', 'a8a7', 'c7f4', 'a7b7', 'f7e5', 'b7g7', 'h7g7', 'f3e2', 'f4d2', 'a3a4', 'h4h3', 'a4a5', 'd2a5', 'f1h3', 'a5c5', 'h3g3', 'g7h7', 'g3h4', 'h7g6', 'h4g3', 'g6f6', ' ']


InvalidMoveError: expected uci string to be of length 4 or 5: ' '